# Dev Assistant — MCP + Gemini

End-to-end agentic app: Gemini picks which tools to call across three MCP servers (filesystem, git, custom dev tools). The LLM drives all tool selection — no hardcoded flow.

**Theme:** Dev Assistant — read/write source files, inspect git history, lint and analyze code.

Run cells top to bottom.

## 1. install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio"

## 2. api key

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key set:", bool(os.environ.get("GOOGLE_API_KEY")))

## 3. check node / npx

In [ ]:
!node --version
!npx --version

In [ ]:
# uncomment if node/npx not found above
# !apt-get -qq update && apt-get -qq install -y nodejs npm
# !node --version && npx --version

## 4. working directory + sample project files

In [ ]:
import asyncio
import nest_asyncio
from pathlib import Path

nest_asyncio.apply()

WORKDIR = "/content/devproject"
Path(WORKDIR).mkdir(exist_ok=True)

(Path(WORKDIR) / "app.py").write_text(
    "import os\n"
    "import sys\n"
    "import json\n"
    "\n"
    "# TODO: add input validation\n"
    "def process_data(data):\n"
    "    result = []\n"
    "    for item in data:\n"
    "        result.append(item * 2)\n"
    "    return result\n"
    "\n"
    "# TODO: handle edge cases\n"
    "def load_config(path):\n"
    "    with open(path) as f:\n"
    "        return json.load(f)\n"
    "\n"
    "def main():\n"
    "    x=1\n"
    "    y=2\n"
    "    print(process_data([x,y,3,4]))\n"
    "\n"
    "if __name__ == '__main__':\n"
    "    main()\n"
)

(Path(WORKDIR) / "utils.py").write_text(
    "# TODO: implement retry logic\n"
    "def fetch_url(url):\n"
    "    pass\n"
    "\n"
    "def parse_response(resp):\n"
    "    if resp is None:\n"
    "        return {}\n"
    "    return resp.json()\n"
)

(Path(WORKDIR) / "README.md").write_text(
    "# devproject\n\nSmall demo project for the dev assistant agent.\n"
)

print("workdir:", WORKDIR)
print("files:", [f.name for f in Path(WORKDIR).iterdir()])

Init a git repo in WORKDIR so the git MCP server has something to work with.

In [ ]:
import subprocess

subprocess.run(["git", "init", WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "config", "user.email", "dev@example.com"], check=True)
subprocess.run(["git", "-C", WORKDIR, "config", "user.name", "Dev"], check=True)
subprocess.run(["git", "-C", WORKDIR, "add", "."], check=True)
subprocess.run(["git", "-C", WORKDIR, "commit", "-m", "initial commit"], check=True)

(Path(WORKDIR) / "app.py").open("a").write("\n# updated\n")
subprocess.run(["git", "-C", WORKDIR, "add", "app.py"], check=True)
subprocess.run(["git", "-C", WORKDIR, "commit", "-m", "update app"], check=True)

print("git repo ready")

## 5. install git MCP server

In [ ]:
%pip install -qU mcp-server-git

## 6. custom MCP server (dev_tools)

Three tools: `count_todos`, `lint_python`, `generate_summary`.

In [ ]:
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List
    import re

    mcp = FastMCP(name="dev_tools")

    @mcp.tool
    def count_todos(content: str) -> Dict[str, int]:
        \"\"\"Count TODO and FIXME comments in source code.\"\"\"
        todos = len(re.findall(r'#\\s*TODO', content, re.IGNORECASE))
        fixmes = len(re.findall(r'#\\s*FIXME', content, re.IGNORECASE))
        return {"todos": todos, "fixmes": fixmes, "total": todos + fixmes}

    @mcp.tool
    def lint_python(code: str) -> List[str]:
        \"\"\"Basic lint checks on Python code. Returns a list of issues found.\"\"\"
        issues = []
        lines = code.splitlines()
        for i, line in enumerate(lines, 1):
            if len(line) > 100:
                issues.append(f"line {i}: line too long ({len(line)} chars)")
            if re.search(r'[a-zA-Z0-9]=[a-zA-Z0-9]', line) and '==' not in line and '>=' not in line and '<=' not in line:
                issues.append(f"line {i}: missing spaces around assignment")
            if 'import *' in line:
                issues.append(f"line {i}: wildcard import")
        if not issues:
            issues.append("no issues found")
        return issues

    @mcp.tool
    def generate_summary(filenames: List[str], todo_counts: Dict[str, int]) -> str:
        \"\"\"Generate a short project health summary from file names and todo counts.\"\"\"
        total = todo_counts.get(\"total\", 0)
        verdict = \"needs attention\" if total > 3 else \"looks okay\"
        return (
            f\"Project has {len(filenames)} file(s): {', '.join(filenames)}.\\n\"
            f\"Found {total} TODO/FIXME comment(s) total — {verdict}.\"
        )

    if __name__ == \"__main__\":
        mcp.run(transport=\"stdio\")
"""), encoding="utf-8")

print("Wrote:", server_path)

## 7. connect to all MCP servers

Third-party servers:
- `@modelcontextprotocol/server-filesystem` — read/write files in WORKDIR
- `mcp-server-git` — git log, diff, status on WORKDIR

Custom:
- `custom_mcp_server.py` — dev_tools (count_todos, lint_python, generate_summary)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "dev_tools": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print("Tool count:", len(tools))
print([t.name for t in tools])

## 8. build the Gemini agent

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
)

async def run_agent(query: str) -> str:
    async with MultiServerMCPClient(mcp_connections) as client:
        tools = await client.get_tools()
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": [("user", query)]})
        return result["messages"][-1].content

def ask(query: str) -> str:
    return asyncio.get_event_loop().run_until_complete(run_agent(query))

print("agent ready")

## 9. run the dev assistant

Each query shows the LLM picking different tool combinations.

### Query 1 — filesystem: list project files

In [ ]:
result = ask("List all files in the dev project directory.")
print(result)

### Query 2 — filesystem + dev_tools: read and lint app.py

In [ ]:
result = ask(
    "Read app.py from the project directory, then run lint_python on its contents "
    "and tell me what issues were found."
)
print(result)

### Query 3 — filesystem + dev_tools: count TODOs across both source files

In [ ]:
result = ask(
    "Read app.py and utils.py from the project directory. "
    "Use count_todos on each file's content and tell me the total TODO/FIXME count for each."
)
print(result)

### Query 4 — git: inspect commit history

In [ ]:
result = ask(
    "Show me the git log for the project and summarize what commits have been made."
)
print(result)

## 10. multi-step demo — full dev audit pipeline

Shows the agent chaining all three servers in one request.

In [ ]:
result = ask(
    "Do these steps in order:\n"
    "1. List all files in the project directory\n"
    "2. Read app.py and utils.py\n"
    "3. Run lint_python on app.py\n"
    "4. Run count_todos on app.py and utils.py (combine the counts)\n"
    "5. Check the git log to see how many commits there are\n"
    "6. Call generate_summary with the list of filenames and the combined todo counts\n"
    "7. Write the result of generate_summary to a file called audit_report.txt"
)
print(result)

In [ ]:
report = Path(WORKDIR + "/audit_report.txt")
if report.exists():
    print(report.read_text())
else:
    print("audit_report.txt not found — check agent output above")